In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# create folder
os.makedirs("charts_final", exist_ok=True)

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("C:\\Users\\96654\\Desktop\\Real Estate\\real_estate_FINAL_fixed (2).xlsx")
print("Data loaded:", df.shape)

# =========================
# SIMPLE CLEANING
# =========================
df['Property_Status'] = df['Year_Built'].apply(
    lambda y: "Off-Plan" if pd.notna(y) and y > 2026 else "Ready"
)

def finishing(val):
    val = str(val)
    if "Ultra" in val: return "Ultra"
    if "Lux" in val: return "Super Lux"
    if "Finished" in val: return "Finished"
    if "Basic" in val: return "Basic"
    return "Other"

df['Finishing_Level'] = df['Amenities'].apply(finishing)

# price tiers
def tier(p):
    if p < 2_000_000: return "Budget"
    elif p < 5_000_000: return "Mid-Range"
    elif p < 10_000_000: return "Premium"
    else: return "Luxury"

df['Price_Tier'] = df['Price_Numeric'].apply(tier)

# area brackets
def area_bracket(a):
    if a < 80: return "Small"
    elif a < 150: return "Medium"
    elif a < 250: return "Large"
    else: return "Very Large"

df["Area_Bracket"] = df["Area"].apply(area_bracket)

# =========================
# SAVE FUNCTION
# =========================
def save(name):
    plt.savefig(f"charts_final/{name}.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved:", name)


Data loaded: (4339, 18)


In [2]:
# Q1 - Expensive vs Cheap Locations
# =====================================================
# Filter English locations only
df_eng = df[df["Location"].str.contains(r'[a-zA-Z]', na=False)]

loc = df_eng.groupby("Location")['Price_Per_m2_Num'].mean().sort_values()

cheap = loc.head(10)
expensive = loc.tail(10)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
cheap.plot(kind='barh', color='green')
plt.title("Cheapest Locations")

plt.subplot(1,2,2)
expensive.plot(kind='barh', color='red')
plt.title("Most Expensive Locations")

plt.tight_layout()
save("Q1_locations")

Saved: Q1_locations


In [3]:
# Q2 - Finishing Impact
# =====================================================
finish = df.groupby("Finishing_Level")['Price_Per_m2_Num'].mean()

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
finish.plot(kind='bar', color='orange')
plt.title("Average Price per m²")

plt.subplot(1,2,2)
sns.boxplot(x='Finishing_Level', y='Price_Per_m2_Num', data=df)
plt.title("Price Distribution by Finishing")

plt.tight_layout()
save("Q2_finishing")

Saved: Q2_finishing


In [4]:
# Q3 - Seller Comparison
# =====================================================
seller = df[df['Seller_Type'].isin(["Broker","Developer"])]

seller_avg = seller.groupby("Seller_Type")['Price_Per_m2_Num'].mean()

plt.figure()
seller_avg.plot(kind='bar', color=['blue','green'])
plt.title("Developer vs Broker Pricing")
save("Q3_seller")

Saved: Q3_seller


In [5]:
# Q4 - Offplan vs Ready
# =====================================================
status = df.groupby("Property_Status")['Price_Per_m2_Num'].mean()

plt.figure()
status.plot(kind='bar', color=['blue','green'])
plt.title("Off-Plan vs Ready Price")
save("Q4_status")


Saved: Q4_status


In [6]:
# Q5 - Correlation
# =====================================================
cols = ['Price_Numeric','Price_Per_m2_Num','Area','Rooms','Bathrooms']
corr = df[cols].corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
save("Q5_corr")


Saved: Q5_corr


In [7]:
# Q6 - Price Tier
# =====================================================
counts = df['Price_Tier'].value_counts()

plt.figure()
counts.plot(kind='pie', autopct='%1.1f%%')
plt.title("Price Tier Distribution")
save("Q6_tier")


Saved: Q6_tier


In [8]:
# Q7 - Undervalued Locations
# =====================================================
avg_price = df['Price_Per_m2_Num'].mean()

cheap_good = df[
    (df['Price_Per_m2_Num'] < avg_price) &
    (df['Finishing_Level'].isin(["Super Lux","Ultra"]))
]

top_cheap = cheap_good.groupby("Location")['Price_Per_m2_Num'].mean().head(10)

plt.figure()
top_cheap.plot(kind='bar', color='purple')
plt.title("Undervalued Locations")
save("Q7_undervalued")

Saved: Q7_undervalued


In [9]:
# Q8 - Price Trend
# =====================================================
trend = df.groupby("Year_Built")['Price_Per_m2_Num'].mean().dropna()

plt.figure()
trend.head(20).plot()
plt.title("Price Trend Over Years")
save("Q8_trend")


Saved: Q8_trend


In [10]:
# Q9 - Outliers
# =====================================================
plt.figure()
sns.boxplot(y=df['Price_Per_m2_Num'])
plt.title("Price Outliers")
save("Q9_outliers")


Saved: Q9_outliers


In [11]:
# Q10 - Area vs Price
# =====================================================
plt.figure()
plt.scatter(df['Area'], df['Price_Numeric']/1e6, alpha=0.5)
plt.title("Area vs Total Price")
plt.xlabel("Area (m²)")
plt.ylabel("Price (Millions EGP)")
save("Q10_scatter")

print("All charts saved in charts_final/")

Saved: Q10_scatter
All charts saved in charts_final/
